In [1]:
import numpyro
import numpyro.distributions as dist
import jax
import jax.numpy as jnp
import numpy as np
from jax import random
from numpyro.infer import Predictive

print(f"NumPyro version : {numpyro.__version__}")
print(f"JAX version     : {jax.__version__}")
print(f"JAX backend     : {jax.default_backend()}")

NumPyro version : 0.21.0
JAX version     : 0.10.0
JAX backend     : cpu


In [2]:
# =============================================================================
# CONFERENCE INDEX MAPS
# Inherited directly from model_01_prior_specification.ipynb Cell 3.
# This list and conf_to_idx must be identical across all notebooks.
# =============================================================================

CONFERENCES = [
    "ACC",               # 0
    "American Athletic", # 1
    "Big 12",            # 2
    "Big Ten",           # 3
    "Conference USA",    # 4
    "Mid-American",      # 5
    "Mountain West",     # 6
    "Pac-12",            # 7
    "SEC",               # 8
    "Sun Belt",          # 9  <- rec_weight hard constraint (non-positive)
]
N_CONFERENCES = len(CONFERENCES)
conf_to_idx   = {c: i for i, c in enumerate(CONFERENCES)}
SUN_BELT_IDX  = conf_to_idx["Sun Belt"]  # 9

print("Conference index map:")
for name, idx in conf_to_idx.items():
    tag = "  <- Sun Belt (rec_weight non-positive)" if idx == SUN_BELT_IDX else ""
    print(f"  {idx:2d} : {name}{tag}")
print(f"\nN_CONFERENCES : {N_CONFERENCES}")
print(f"SUN_BELT_IDX  : {SUN_BELT_IDX}")
print("Cell 2 complete.")

Conference index map:
   0 : ACC
   1 : American Athletic
   2 : Big 12
   3 : Big Ten
   4 : Conference USA
   5 : Mid-American
   6 : Mountain West
   7 : Pac-12
   8 : SEC
   9 : Sun Belt  <- Sun Belt (rec_weight non-positive)

N_CONFERENCES : 10
SUN_BELT_IDX  : 9
Cell 2 complete.


In [3]:
# =============================================================================
# CONFERENCE-SCOPE MASK BUILDER
# Source: artifacts/final_features.csv conference_scope column.
#
# Design: one coefficient per conference-scoped feature, one mask per feature.
# Masks are binary float32 arrays of length N_games, built from the team's
# conference before model_cfb() is called. Inside the model the term is:
#   beta * mask[i] * feature_value[i]
# mask=0 -> term contributes zero regardless of feature value.
# No separate priors per conference for any game-level feature.
# =============================================================================

# Conference sets — exactly as listed in final_features.csv conference_scope
SCOPE_LAST3_OFF_EPA      = {"ACC", "Mid-American", "SEC"}
SCOPE_LAST3_DEF_EPA      = {"American Athletic", "Big Ten", "Conference USA",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_LAST3_PTS_SCORED   = {"ACC", "Big 12", "Big Ten", "Conference USA",
                             "Mid-American", "Mountain West"}
SCOPE_LAST3_PTS_ALLOWED  = {"American Athletic", "Big Ten", "Conference USA",
                             "Mountain West", "Pac-12", "Sun Belt"}
SCOPE_DAYS_SINCE         = {"American Athletic", "Big 12"}
SCOPE_CLOSE_PLAY_COUNT   = {"ACC", "American Athletic", "Big 12",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_ELEVATION          = {"Mountain West", "Big 12"}


def build_conf_mask(team_conferences, scope_set):
    """
    Build a binary mask array of shape (N_games,).

    Parameters
    ----------
    team_conferences : list of str
        Conference name for each game-row team.
    scope_set : set of str
        Confirmed conference list for this feature.

    Returns
    -------
    jnp.ndarray float32, shape (N_games,)
        1.0 where team conference is in scope_set, 0.0 otherwise.
    """
    return jnp.array(
        [1.0 if c in scope_set else 0.0 for c in team_conferences],
        dtype=jnp.float32
    )


# --- Smoke test ---
_test_confs    = ["ACC", "Big Ten", "SEC", "Sun Belt", "Mountain West", "Pac-12"]
_test_mask     = build_conf_mask(_test_confs, SCOPE_LAST3_OFF_EPA)
_expected      = jnp.array([1., 0., 1., 0., 0., 0.])
assert jnp.allclose(_test_mask, _expected), f"Mask mismatch: {_test_mask}"

print("Conference-scope mask builder verified.")
print(f"  Test conferences : {_test_confs}")
print(f"  SCOPE_LAST3_OFF_EPA : {sorted(SCOPE_LAST3_OFF_EPA)}")
print(f"  Mask result         : {_test_mask.tolist()}")
print(f"  Expected            : {_expected.tolist()}")
print()
print("Scoped features defined:")
scoped = {
    "last3_off_epa_avg"         : SCOPE_LAST3_OFF_EPA,
    "last3_def_epa_avg"         : SCOPE_LAST3_DEF_EPA,
    "last3_points_scored_avg"   : SCOPE_LAST3_PTS_SCORED,
    "last3_points_allowed_avg"  : SCOPE_LAST3_PTS_ALLOWED,
    "days_since_last_game"      : SCOPE_DAYS_SINCE,
    "close_game_play_count_delta": SCOPE_CLOSE_PLAY_COUNT,
    "away_elevation_delta_ft"   : SCOPE_ELEVATION,
}
for fname, scope in scoped.items():
    print(f"  {fname:<35s}: {sorted(scope)}")
print("Cell 3 complete.")

Conference-scope mask builder verified.
  Test conferences : ['ACC', 'Big Ten', 'SEC', 'Sun Belt', 'Mountain West', 'Pac-12']
  SCOPE_LAST3_OFF_EPA : ['ACC', 'Mid-American', 'SEC']
  Mask result         : [1.0, 0.0, 1.0, 0.0, 0.0, 0.0]
  Expected            : [1.0, 0.0, 1.0, 0.0, 0.0, 0.0]

Scoped features defined:
  last3_off_epa_avg                  : ['ACC', 'Mid-American', 'SEC']
  last3_def_epa_avg                  : ['American Athletic', 'Big Ten', 'Conference USA', 'Mid-American', 'Pac-12', 'Sun Belt']
  last3_points_scored_avg            : ['ACC', 'Big 12', 'Big Ten', 'Conference USA', 'Mid-American', 'Mountain West']
  last3_points_allowed_avg           : ['American Athletic', 'Big Ten', 'Conference USA', 'Mountain West', 'Pac-12', 'Sun Belt']
  days_since_last_game               : ['American Athletic', 'Big 12']
  close_game_play_count_delta        : ['ACC', 'American Athletic', 'Big 12', 'Mid-American', 'Pac-12', 'Sun Belt']
  away_elevation_delta_ft            : ['Big 12', 

In [4]:
# =============================================================================
# GAME DATA CONTAINER
# A single object passed to model_cfb() instead of 20+ positional arguments.
# All arrays have length N_games unless noted.
#
# Threshold-activated features (elevation, travel, timezone, wind, bye week)
# arrive pre-zeroed from the data layer when threshold is not met.
# The model treats them as regular covariates — no in-model threshold logic.
#
# Conference-scoped features arrive with a paired mask built by
# build_conf_mask(). The model applies: beta * mask * feature_value.
#
# Archetype fields are int32 index arrays (values 0-3), NOT float arrays.
# The model indexes into a 4-vector embedding: b_off_archetype[off_archetype_idx]
# Do NOT pass float archetypes or 16-way compound matchup strings.
# =============================================================================

from dataclasses import dataclass
from typing import Optional


@dataclass
class GameData:
    # Index arrays (int32)
    team_idx  : jnp.ndarray  # [0, N_teams)
    opp_idx   : jnp.ndarray  # [0, N_teams)
    conf_idx  : jnp.ndarray  # [0, N_CONFERENCES) — team's conference

    # Home indicator
    is_home   : jnp.ndarray  # float32 {0.0, 1.0}

    # Observed scores — None for prior predictive
    points    : Optional[jnp.ndarray]  # int32 or None

    # Prior seeds (team-level, not game-level predictors)
    sp_rating          : jnp.ndarray  # float32, standardized
    recruiting_3yr_avg : jnp.ndarray  # float32, standardized

    # Universal game-level features (active in all conferences)
    close_game_epa        : jnp.ndarray  # float32, standardized
    close_game_def_epa    : jnp.ndarray  # float32, standardized
    pregame_elo           : jnp.ndarray  # float32, standardized
    elo_sp_divergence     : jnp.ndarray  # float32, already z-score (do not re-standardize)
    last3_win_pct         : jnp.ndarray  # float32, standardized; imputed with season prior when null
    away_travel_distance  : jnp.ndarray  # float32, div by non-zero std; pre-zeroed when < 1500mi
    away_tz_delta         : jnp.ndarray  # float32, div by non-zero std; pre-zeroed when abs < 2hr
    wind_chill            : jnp.ndarray  # float32, div by non-zero std; pre-zeroed when > 40F or is_dome
    rush_rate_std_downs   : jnp.ndarray  # float32, standardized
    rush_rate_pass_downs  : jnp.ndarray  # float32, standardized
    off_sack_rate_allowed : jnp.ndarray  # float32, standardized
    def_sack_rate         : jnp.ndarray  # float32, standardized

    # Archetype index arrays — int32, values 0-3
    # Embedded in model as b_off_archetype[off_archetype_idx]
    # Source: off_archetype_id and def_archetype_id columns (EDA 10)
    # Do NOT use 16-way compound matchup string columns
    off_archetype_idx : jnp.ndarray  # int32
    def_archetype_idx : jnp.ndarray  # int32

    # Conference-scoped features + masks (7 pairs)
    last3_off_epa_avg      : jnp.ndarray
    mask_last3_off_epa     : jnp.ndarray  # 1 if ACC/Mid-American/SEC
    last3_def_epa_avg      : jnp.ndarray
    mask_last3_def_epa     : jnp.ndarray  # 1 if AmAth/BigTen/CUSA/MAC/Pac12/SunBelt
    last3_pts_scored_avg   : jnp.ndarray
    mask_last3_pts_scored  : jnp.ndarray  # 1 if ACC/Big12/BigTen/CUSA/MAC/MWC
    last3_pts_allowed_avg  : jnp.ndarray
    mask_last3_pts_allowed : jnp.ndarray  # 1 if AmAth/BigTen/CUSA/MWC/Pac12/SunBelt
    days_since_last_game   : jnp.ndarray  # float32, div by non-zero std; pre-zeroed when < 12 days
    mask_days_since        : jnp.ndarray  # 1 if AmAth/Big12
    close_play_count_delta : jnp.ndarray
    mask_close_play_count  : jnp.ndarray  # 1 if ACC/AmAth/Big12/MAC/Pac12/SunBelt
    away_elevation_delta   : jnp.ndarray  # float32, div by non-zero std; pre-zeroed when < 2000ft
    mask_elevation         : jnp.ndarray  # 1 if MWC/Big12


print("GameData container defined.")
print("  Universal game-level features : 14  (continuous float32)")
print("  Archetype index fields        :  2  (int32, values 0-3)")
print("  Conference-scoped features    :  7  (+ 7 masks)")
print("  Prior seeds                   :  2")
print("  Total model features          : 23  (matches final_features.csv)")
print("Cell 4 complete.")

GameData container defined.
  Universal game-level features : 14  (continuous float32)
  Archetype index fields        :  2  (int32, values 0-3)
  Conference-scoped features    :  7  (+ 7 masks)
  Prior seeds                   :  2
  Total model features          : 23  (matches final_features.csv)
Cell 4 complete.


In [5]:
# =============================================================================
# model_cfb() — full hierarchical Negative Binomial model
#
# Three-level hierarchy: league -> conference -> team
# Likelihood: Negative Binomial 2
#
# All parameter names match model_01_prior_specification.ipynb exactly.
# model_04 passes this function to numpyro.infer.MCMC. No fitting here.
#
# All parameters are additive on the log scale.
# mu_league  : Normal(3.3, 0.2)   — exp(3.3) ≈ 27 pts
# hfa_league : Normal(0.1, 0.05)  — ≈ 2.43 pts on 27 pt baseline
# sigma_hfa_team : HalfNormal(0.1) — log scale team HFA deviation
# =============================================================================

def model_cfb(data: GameData, N_teams: int):
    """
    Hierarchical Negative Binomial model for CFB scoring.

    Parameters
    ----------
    data    : GameData — all index and feature arrays for this batch
    N_teams : int      — unique team×season combinations in training data

    When data.points is None (prior predictive), numpyro samples freely
    from the likelihood without conditioning on observations.
    """

    # =========================================================================
    # LEAGUE LEVEL
    # Prior spec sections 1.1, 1.2, 1.3
    # =========================================================================

    # 1.1 League intercept — log scale; exp(3.3) ≈ 27 pts (FBS mean 2022-2024)
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))

    # 1.2 HFA baseline — log scale; 0.1 ≈ 2.43 pts on 27 pt baseline
    # EDA 05: league-level HFA = +2.43 pts (p<0.001)
    # No conference-level HFA layer; team deviations handle within-conf spread
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    # 1.3 Dispersion — conference-specific vector, length N_CONFERENCES
    # EDA 05: Bartlett p=0.000470, Levene p=0.000705 — conference dispersion
    #   differences statistically confirmed; scalar r caused sampler collapse
    # EDA 01: VMR range 4.95-7.16 across conferences
    # Gamma(2.0, 0.1): mean=20, weakly informative; each conference independent
    r_negbinom = numpyro.sample(
        "r_negbinom",
        dist.Gamma(2.0, 0.1),
        sample_shape=(N_CONFERENCES,)
    )

    # =========================================================================
    # CONFERENCE LEVEL
    # Prior spec section 2.1
    # EDA 05: conference ICC 0.02-0.05; pooling improves small-sample estimates
    # mu_conference[conf_idx[i]] enters log(mu) as an additive intercept offset
    # =========================================================================

    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(3.0))
    mu_conference    = numpyro.sample(
        "mu_conference",
        dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    # =========================================================================
    # TEAM LEVEL
    # Prior spec sections 3.1, 3.2, 3.3, 3.4, 3.5
    # Non-centered parameterization used here for alpha_team, delta_team,
    # hfa_team to improve sampler geometry in model_04
    # =========================================================================

    # 3.1 Attack — log-scale offensive strength
    sigma_attack = numpyro.sample("sigma_attack", dist.HalfNormal(0.4))
    alpha_team   = numpyro.sample(
        "alpha_team",
        dist.Normal(0.0, sigma_attack),
        sample_shape=(N_teams,)
    )

    # 3.2 Defense — enters log(mu) as NEGATIVE (stronger defense lowers opp mu)
    sigma_defense = numpyro.sample("sigma_defense", dist.HalfNormal(0.4))
    delta_team    = numpyro.sample(
        "delta_team",
        dist.Normal(0.0, sigma_defense),
        sample_shape=(N_teams,)
    )

    # 3.3 Team HFA deviation from league hfa_league
    # EDA 05: team HFA SD = 4.81 pts; 4.81/27 ≈ 0.18 on log scale
    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team       = numpyro.sample(
        "hfa_team",
        dist.Normal(0.0, sigma_hfa_team),
        sample_shape=(N_teams,)
    )

    # 3.4 SP+ prior seed — DUAL ROLE: prior seed AND game-level spread predictor
    # EDA 04: YoY r=0.7632; partial r=0.197 after EPA control, p<0.0001
    # Signal holds through games 9-12 (r=0.2609) — do not decay aggressively
    # sp_offense/sp_defense excluded — DATA LEAKAGE (end-of-season values)
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 1.0))

    # 3.5 Recruiting prior seed — conference-specific; Sun Belt non-positive
    # EDA 04: YoY r=0.9779; TruncatedNormal(high=0) for Sun Belt (idx=9)
    # Sun Belt: rec<->diff_r=-0.2665; positive weight would introduce
    #   systematic error
    rec_weight_other   = numpyro.sample(
        "rec_weight_other",
        dist.Normal(0.0, 0.5),
        sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt",
        dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    # =========================================================================
    # GAME-LEVEL FEATURE COEFFICIENTS
    # All names match model_01_prior_specification.ipynb Cell 5 exactly
    # =========================================================================

    # Anchors — prior spec 4.1
    b_close_game_epa     = numpyro.sample("b_close_game_epa",     dist.Normal(0.0, 0.5))
    b_close_game_def_epa = numpyro.sample("b_close_game_def_epa", dist.Normal(0.0, 0.5))

    # ELO — prior spec 4.2, 4.3
    # elo_sp_divergence: r=-0.1150 (negative direction, z-score version, EDA 08 corrected)
    b_pregame_elo        = numpyro.sample("b_pregame_elo",        dist.Normal(0.0, 0.3))
    b_elo_sp_divergence  = numpyro.sample("b_elo_sp_divergence",  dist.Normal(0.0, 0.2))

    # Momentum universal — prior spec 4.4
    b_last3_win_pct      = numpyro.sample("b_last3_win_pct",      dist.Normal(0.0, 0.3))

    # Environmental — prior spec 4.5; values pre-zeroed when threshold not met
    b_away_travel_distance = numpyro.sample("b_away_travel_distance", dist.Normal(0.0, 0.3))
    b_away_tz_delta        = numpyro.sample("b_away_tz_delta",        dist.Normal(0.0, 0.3))
    b_wind_chill           = numpyro.sample("b_wind_chill",           dist.Normal(0.0, 0.2))

    # Style/tempo — prior spec 4.6
    b_rush_rate_std_downs  = numpyro.sample("b_rush_rate_std_downs",  dist.Normal(0.0, 0.3))
    b_rush_rate_pass_downs = numpyro.sample("b_rush_rate_pass_downs", dist.Normal(0.0, 0.3))

    # Sack-rate mismatch — prior spec 4.7
    b_off_sack_rate_allowed = numpyro.sample("b_off_sack_rate_allowed", dist.Normal(0.0, 0.2))
    b_def_sack_rate         = numpyro.sample("b_def_sack_rate",         dist.Normal(0.0, 0.2))

    # Archetype embeddings — prior spec 4.8
    # 4-vector per archetype type; indexed by int32 arrays (values 0-3)
    # EDA 10: defense matchup eta²=0.39 on O/U; weak spread signal (eta²=0.021)
    # Deployment condition: pregame rolling version required before 2026-09-24
    b_off_archetype = numpyro.sample("b_off_archetype",
                                      dist.Normal(0.0, 0.3),
                                      sample_shape=(4,))
    b_def_archetype = numpyro.sample("b_def_archetype",
                                      dist.Normal(0.0, 0.3),
                                      sample_shape=(4,))

    # Conference-scoped — one coefficient each, masks handle zeroing
    # prior spec 4.4, 4.9
    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.3))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.3))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.3))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.3))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.2))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.2))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.3))

    # =========================================================================
    # LINEAR PREDICTOR — log(mu)
    #
    # log(mu) = mu_league
    #         + mu_conference[conf_idx]             conference offset
    #         + alpha_team[team_idx]                team attack
    #         - delta_team[opp_idx]                 opponent defense
    #         + (hfa_league + hfa_team[team_idx]) * is_home
    #         + sp_weight * sp_rating
    #         + rec_weight[conf_idx] * recruiting_3yr_avg
    #         + [14 universal continuous game-level terms]
    #         + b_off_archetype[off_archetype_idx]  embedding lookup
    #         + b_def_archetype[def_archetype_idx]  embedding lookup
    #         + [7 conference-scoped terms x masks]
    # =========================================================================

    log_mu = (
        # League baseline
        mu_league

        # Conference offset
        + mu_conference[data.conf_idx]

        # Team attack
        + alpha_team[data.team_idx]

        # Opponent defense — subtracted: stronger defense lowers opponent mu
        - delta_team[data.opp_idx]

        # Home field: league baseline + team deviation
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home

        # SP+ prior seed — also game-level spread predictor (partial r=0.197)
        + sp_weight * data.sp_rating

        # Recruiting prior seed — conf-indexed; Sun Belt guaranteed <= 0
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg

        # --- Universal continuous game-level features ---

        + b_close_game_epa      * data.close_game_epa
        + b_close_game_def_epa  * data.close_game_def_epa
        + b_pregame_elo         * data.pregame_elo
        + b_elo_sp_divergence   * data.elo_sp_divergence
        + b_last3_win_pct       * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate

        # --- Archetype embeddings — index lookup, not scalar multiply ---
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]

        # --- Conference-scoped features x masks ---
        # Pattern: beta * mask * feature_value
        # mask=0 -> zero contribution for out-of-scope conferences

        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )

    # =========================================================================
    # LIKELIHOOD
    # mu = exp(log_mu) — expected points scored
    # NegativeBinomial2(mean=mu, concentration=r_negbinom[conf_idx])
    # Conference-specific r selected by team's conference index
    # data.points=None during prior predictive — numpyro samples freely
    # =========================================================================

    mu = jnp.exp(log_mu)

    numpyro.sample(
        "obs",
        dist.NegativeBinomial2(mu, r_negbinom[data.conf_idx]),
        obs=data.points
    )


print("model_cfb() defined.")
print("  r_negbinom         : Gamma(2.0, 0.1) x N_CONFERENCES  [conference-specific]")
print("  b_off_archetype    : Normal(0.0, 0.3) x 4  [embedding, indexed by off_archetype_idx]")
print("  b_def_archetype    : Normal(0.0, 0.3) x 4  [embedding, indexed by def_archetype_idx]")
print("  Likelihood         : NegativeBinomial2(mu, r_negbinom[conf_idx])")
print("Cell 5 complete.")

model_cfb() defined.
  r_negbinom         : Gamma(2.0, 0.1) x N_CONFERENCES  [conference-specific]
  b_off_archetype    : Normal(0.0, 0.3) x 4  [embedding, indexed by off_archetype_idx]
  b_def_archetype    : Normal(0.0, 0.3) x 4  [embedding, indexed by def_archetype_idx]
  Likelihood         : NegativeBinomial2(mu, r_negbinom[conf_idx])
Cell 5 complete.


In [6]:
# =============================================================================
# CELL 6 — Structural verification
# Prior predictive draw on synthetic data — confirms model_cfb() samples all
# parameters correctly and produces plausible score ranges.
# No MCMC, no fitting, no real data.
# N_TEAMS_TEST set to 10 for speed; model_04 replaces with actual training count.
# =============================================================================

from numpyro.infer import Predictive

N_TEAMS_TEST = 10
N_GAMES_TEST = 20

# Synthetic index arrays
rng      = np.random.default_rng(42)
team_idx = jnp.array(rng.integers(0, N_TEAMS_TEST, N_GAMES_TEST), dtype=jnp.int32)
opp_idx  = jnp.array(rng.integers(0, N_TEAMS_TEST, N_GAMES_TEST), dtype=jnp.int32)
conf_idx = jnp.array(rng.integers(0, N_CONFERENCES, N_GAMES_TEST), dtype=jnp.int32)
is_home  = jnp.array(rng.integers(0, 2, N_GAMES_TEST), dtype=jnp.float32)

# Archetype index arrays — int32, values 0-3
off_archetype_idx = jnp.array(rng.integers(0, 4, N_GAMES_TEST), dtype=jnp.int32)
def_archetype_idx = jnp.array(rng.integers(0, 4, N_GAMES_TEST), dtype=jnp.int32)

# Conference names for mask building
conf_names = [CONFERENCES[i] for i in conf_idx.tolist()]

# All continuous feature arrays zero — sufficient for structural verification
zeros = jnp.zeros(N_GAMES_TEST, dtype=jnp.float32)

synthetic_data = GameData(
    team_idx  = team_idx,
    opp_idx   = opp_idx,
    conf_idx  = conf_idx,
    is_home   = is_home,
    points    = None,

    sp_rating          = zeros,
    recruiting_3yr_avg = zeros,

    close_game_epa        = zeros,
    close_game_def_epa    = zeros,
    pregame_elo           = zeros,
    elo_sp_divergence     = zeros,
    last3_win_pct         = zeros,
    away_travel_distance  = zeros,
    away_tz_delta         = zeros,
    wind_chill            = zeros,
    rush_rate_std_downs   = zeros,
    rush_rate_pass_downs  = zeros,
    off_sack_rate_allowed = zeros,
    def_sack_rate         = zeros,

    off_archetype_idx = off_archetype_idx,
    def_archetype_idx = def_archetype_idx,

    last3_off_epa_avg      = zeros,
    mask_last3_off_epa     = build_conf_mask(conf_names, SCOPE_LAST3_OFF_EPA),
    last3_def_epa_avg      = zeros,
    mask_last3_def_epa     = build_conf_mask(conf_names, SCOPE_LAST3_DEF_EPA),
    last3_pts_scored_avg   = zeros,
    mask_last3_pts_scored  = build_conf_mask(conf_names, SCOPE_LAST3_PTS_SCORED),
    last3_pts_allowed_avg  = zeros,
    mask_last3_pts_allowed = build_conf_mask(conf_names, SCOPE_LAST3_PTS_ALLOWED),
    days_since_last_game   = zeros,
    mask_days_since        = build_conf_mask(conf_names, SCOPE_DAYS_SINCE),
    close_play_count_delta = zeros,
    mask_close_play_count  = build_conf_mask(conf_names, SCOPE_CLOSE_PLAY_COUNT),
    away_elevation_delta   = zeros,
    mask_elevation         = build_conf_mask(conf_names, SCOPE_ELEVATION),
)

# Draw 5 prior predictive samples
predictive    = Predictive(model_cfb, num_samples=5)
prior_samples = predictive(random.PRNGKey(0), data=synthetic_data, N_teams=N_TEAMS_TEST)

# Verify parameter count and shapes
print("Prior predictive draw complete.")
print(f"  Parameters sampled : {len(prior_samples)}")
print()
for name, val in sorted(prior_samples.items()):
    print(f"  {name:<30s} shape={str(val.shape):<20s} sample={float(val.flatten()[0]):.4f}")

# Key assertions
assert prior_samples["obs"].shape == (5, N_GAMES_TEST), \
    f"obs shape wrong: {prior_samples['obs'].shape}"

assert prior_samples["r_negbinom"].shape == (5, N_CONFERENCES), \
    f"r_negbinom shape wrong: {prior_samples['r_negbinom'].shape} — expected (5, {N_CONFERENCES})"

assert prior_samples["b_off_archetype"].shape == (5, 4), \
    f"b_off_archetype shape wrong: {prior_samples['b_off_archetype'].shape}"

assert prior_samples["b_def_archetype"].shape == (5, 4), \
    f"b_def_archetype shape wrong: {prior_samples['b_def_archetype'].shape}"

assert prior_samples["mu_conference"].shape == (5, N_CONFERENCES), \
    f"mu_conference shape wrong: {prior_samples['mu_conference'].shape}"

assert prior_samples["alpha_team"].shape == (5, N_TEAMS_TEST), \
    f"alpha_team shape wrong: {prior_samples['alpha_team'].shape}"

assert float(prior_samples["rec_weight_sunbelt"].max()) <= 0.0, \
    "Sun Belt constraint violated"

# Check no stale parameter names present
stale_names = {"b_offense_archetype", "b_defense_archetype",
               "b_home_off_away_def", "b_away_off_home_def"}
found_stale = stale_names & set(prior_samples.keys())
assert not found_stale, f"Stale parameters still present: {found_stale}"

obs = prior_samples["obs"]
print(f"\nObserved score samples (shape {obs.shape}):")
print(f"  min={int(obs.min())}  max={int(obs.max())}  mean={float(obs.mean()):.1f}")
print(f"\nr_negbinom shape         : {prior_samples['r_negbinom'].shape}  (must be (5, {N_CONFERENCES}))")
print(f"b_off_archetype shape    : {prior_samples['b_off_archetype'].shape}  (must be (5, 4))")
print(f"b_def_archetype shape    : {prior_samples['b_def_archetype'].shape}  (must be (5, 4))")
print(f"Sun Belt rec_weight max  : {float(prior_samples['rec_weight_sunbelt'].max()):.4f}  (must be <= 0)")
print("\nAll assertions passed.")
print("Cell 6 complete.")

Prior predictive draw complete.
  Parameters sampled : 36

  alpha_team                     shape=(5, 10)              sample=-0.0898
  b_away_elevation_delta         shape=(5,)                 sample=-0.0985
  b_away_travel_distance         shape=(5,)                 sample=-0.0961
  b_away_tz_delta                shape=(5,)                 sample=0.2738
  b_close_game_def_epa           shape=(5,)                 sample=1.1831
  b_close_game_epa               shape=(5,)                 sample=-0.6199
  b_close_play_count_delta       shape=(5,)                 sample=-0.0471
  b_days_since_last_game         shape=(5,)                 sample=-0.1181
  b_def_archetype                shape=(5, 4)               sample=0.2984
  b_def_sack_rate                shape=(5,)                 sample=-0.2381
  b_elo_sp_divergence            shape=(5,)                 sample=0.0855
  b_last3_def_epa_avg            shape=(5,)                 sample=0.0343
  b_last3_off_epa_avg            shape=(5,)   

## Notebook Summary — model_02_architecture.ipynb

### What This Notebook Does

This notebook writes the full hierarchical Negative Binomial model function `model_cfb()` for the CFB scoring model. It does not fit the model, does not run MCMC, and does not touch any data. `model_04_first_fit.ipynb` passes `model_cfb()` to `numpyro.infer.MCMC` and fits on 2022–2024 training data for the first time.

### What model_02 Adds That model_01 Does Not Have

`model_01_prior_specification.ipynb` defined every prior distribution in isolation — it called `numpyro.sample()` for each parameter and verified the distributions were valid. It has no likelihood statement and no conditioning on data.

model_02 adds four things model_01 cannot do:

1. **Data interface** — `model_cfb()` accepts a `GameData` object containing all index arrays and feature arrays for a batch of games.
2. **Linear predictor** — wires all sampled parameters into `log(mu)` following the locked equation from the prior specification document.
3. **Conference-scope masking** — precomputed binary masks zero out conference-scoped feature contributions for teams outside the confirmed conference list. One prior per feature; masking handles the zeroing without separate priors per conference.
4. **Likelihood** — `numpyro.sample("obs", NegativeBinomial2(mu, r_negbinom[conf_idx]), obs=data.points)` conditions the model on observed scores. When `data.points=None` the model samples freely for prior predictive checks.

### Three-Level Hierarchy and How It Enters log(mu)

The model pools information across three levels:

- **League** — global baseline: `mu_league`, `hfa_league`, and `r_negbinom` — a vector of length 10, one dispersion parameter per conference
- **Conference** — ten additive offsets from the league baseline: `mu_conference[conf_idx]` selects the learned offset for the team's conference and adds it directly to `log(mu)`. The scale `sigma_conference` is itself learned, providing partial pooling.
- **Team** — individual attack (`alpha_team`), defense (`delta_team`), and HFA deviation (`hfa_team`) for each team×season, pooled within conference via learned hyperprior scales.

### Conference-Specific Dispersion

`r_negbinom` is a vector of length `N_CONFERENCES` — one dispersion value per conference. EDA 05 confirmed statistically significant dispersion differences across conferences (Bartlett p=0.000470, Levene p=0.000705). The likelihood selects the correct conference's dispersion for each game row via `r_negbinom[data.conf_idx]`. Prior: `Gamma(2.0, 0.1)` per conference independently.

### Conference-Scope Masking

Seven features are active only within a confirmed subset of conferences (e.g. `last3_off_epa_avg` only in ACC, Mid-American, SEC). One coefficient is sampled per feature. A binary mask array of length `N_games` is built before `model_cfb()` is called using `build_conf_mask()` — `1.0` where the team's conference is in scope, `0.0` otherwise. Inside the model the term is `beta * mask * feature_value`. When `mask=0` the term contributes nothing regardless of the feature value.

### Archetype Encoding

Style archetypes (EDA 10) are 4-level categorical variables. Each is embedded as a 4-vector — `b_off_archetype` and `b_def_archetype`, each with `sample_shape=(4,)`. The linear predictor indexes into these vectors directly: `b_off_archetype[data.off_archetype_idx]` and `b_def_archetype[data.def_archetype_idx]`. The `GameData` fields `off_archetype_idx` and `def_archetype_idx` must be `int32` arrays with values 0–3. The 16-way compound matchup string columns are not used.

### What model_03 Does

`model_03_feature_engineering.ipynb` loads the 2022–2024 training data from the database, builds all index arrays and feature matrices, computes archetype assignments, applies the preprocessing rules (standardization, sparse threshold-activated scaling, z-score passthrough for `elo_sp_divergence`), writes `scaler_stats.json`, and constructs the `GameData` object that model_04 passes to `model_cfb()`.